# Notebook 3 — Data Packaging for Pretraining
**Adapted for Jupyter Notebook / Kubeflow**

Loads the cleaned dataset from Notebook 2, tokenises it, and packs token sequences
into fixed-length blocks. Saves the result as `./data/packaged_pretrain_dataset.parquet`.

**Requires:** `./data/preprocessed_dataset.parquet` from Notebook 2

**Learning objectives:**
1. Load and shard a cleaned dataset
2. Tokenise text with BOS/EOS markers using SmolLM2's tokenizer
3. Pack variable-length sequences into fixed-length blocks for GPU efficiency
4. Save the packed dataset for the training loop

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "datasets", "transformers", "-q"])

In [ ]:
import os
import datasets
import numpy as np

# ── Data directory — same as used in Notebook 2 ──────────────────────
data_dir = "./data"

input_path = os.path.join(data_dir, "preprocessed_dataset.parquet")
if not os.path.exists(input_path):
    raise FileNotFoundError(
        f"Input file not found: {input_path}\n"
        "Run Notebook 2 first to generate preprocessed_dataset.parquet"
    )
print(f"✅ Input file found: {input_path}")

## 1. Tokenising and Creating input_ids

### 1a. Load the Cleaned Dataset

In [ ]:
dataset = datasets.load_dataset(
    "parquet",
    data_files=input_path,
    split="train"
)
print(dataset)
print(f"\nLoaded {dataset.num_rows:,} cleaned documents")

### 1b. Shard the Dataset

**Sharding** splits the dataset into N equal pieces for parallel distributed training.
Each GPU worker processes one shard. Here we use shard 0 to keep the demo fast.

In [ ]:
# Split into 10 shards — use only shard 0 for the demo
dataset = dataset.shard(num_shards=10, index=0)
print(f"Shard 0 size: {dataset.num_rows:,} rows (1/10 of the full dataset)")
print(dataset)

### 1c. Load the Tokenizer

We use **SmolLM2-360M**'s tokenizer — BPE, vocab size 49,152.
This MUST match the model used in Notebook 4 and 5.

In [ ]:
from transformers import AutoTokenizer

# SmolLM2 tokenizer — must match the model in Notebooks 4 and 5
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")

print(f"Tokenizer   : SmolLM2  |  Vocab size: {tokenizer.vocab_size:,}")
print(f"BOS token   : '{tokenizer.bos_token}' (id={tokenizer.bos_token_id})")
print(f"EOS token   : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

# Quick sanity check
sample_text = "I'm a short sentence"
tokens      = tokenizer.tokenize(sample_text)
print(f"\nSample: '{sample_text}'")
print(f"Tokens : {tokens}")
print(f"IDs    : {tokenizer.convert_tokens_to_ids(tokens)}")

### 1d. Tokenise the Dataset

Each document gets:
1. Tokenised into subword tokens
2. Converted to integer IDs
3. Wrapped with **BOS** (Beginning Of Sequence) and **EOS** (End Of Sequence) markers

> BOS/EOS tokens tell the model where one document ends and the next begins within a packed sequence.

In [ ]:
def tokenization(example):
    tokens    = tokenizer.tokenize(example["text"])
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    # Wrap with document boundary markers
    token_ids = [tokenizer.bos_token_id] + token_ids + [tokenizer.eos_token_id]
    example["input_ids"]  = token_ids
    example["num_tokens"] = len(token_ids)
    return example

dataset = dataset.map(tokenization, load_from_cache_file=False)
print(dataset)

In [ ]:
# Inspect a sample
sample = dataset[3]
print("Text (first 60 chars) :", sample["text"][:60])
print("input_ids (first 15)  :", sample["input_ids"][:15])
print("num_tokens            :", sample["num_tokens"])

# Total token count
total_tokens = np.sum(dataset["num_tokens"])
print(f"\nTotal tokens in shard : {total_tokens:,}")
print(f"Approx. equivalent    : {total_tokens / 1e6:.2f}M tokens")

## 2. Packing the Data

**Why pack?**

Pretraining uses fixed-length input sequences matching the model's context window.
Documents vary widely in length — some 50 tokens, others 2000.

- **Naive approach:** pad each document to `max_seq_length` → wastes compute on padding tokens (zeros carry no information)
- **Packing approach:** concatenate all token IDs into one long stream, then slice into fixed-length chunks → ~100% GPU utilisation, zero wasted tokens

> `max_seq_length = 32` here for visibility. Real models use 512–8192. The principle is identical.

In [ ]:
# Concatenate all token IDs into one long stream
input_ids = np.concatenate(dataset["input_ids"])
print(f"Total token stream length: {len(input_ids):,}")

# Context window size — small for demo clarity
max_seq_length = 32
print(f"Packing into chunks of {max_seq_length} tokens")

# Trim to exact multiple of max_seq_length (drop last incomplete chunk)
total_length = len(input_ids) - len(input_ids) % max_seq_length
print(f"Trimmed length : {total_length:,}")
print(f"Tokens dropped : {len(input_ids) - total_length}")

In [ ]:
input_ids = input_ids[:total_length]

# Reshape into (num_chunks, max_seq_length)
input_ids_reshaped = input_ids.reshape(-1, max_seq_length).astype(np.int32)
print(f"Packed shape: {input_ids_reshaped.shape}")
print(f"→ {input_ids_reshaped.shape[0]:,} training sequences of {max_seq_length} tokens each")

In [ ]:
# Inspect the first packed sequence
print("First packed sequence (token IDs):")
print(input_ids_reshaped[0])
print("\nDecoded:")
print(tokenizer.decode(input_ids_reshaped[0].tolist()))

# Note: decoded output contains fragments from multiple documents — this is expected!
# BOS/EOS tokens mark the document boundaries within the packed sequence.

## 3. Save the Packed Dataset

In [ ]:
input_ids_list = input_ids_reshaped.tolist()
packaged_pretrain_dataset = datasets.Dataset.from_dict({"input_ids": input_ids_list})
print(packaged_pretrain_dataset)

# ── Save to ./data/ ───────────────────────────────────────────────────
output_path = os.path.join(data_dir, "packaged_pretrain_dataset.parquet")
packaged_pretrain_dataset.to_parquet(output_path)
print(f"\n✅ Saved to: {output_path}")

In [ ]:
# Reload and verify
verify = datasets.load_dataset("parquet", data_files=output_path, split="train")
print(f"✅ Verified: {verify.num_rows:,} packed sequences of length {len(verify[0]['input_ids'])}")

## Summary

| Concept | Detail |
|---|---|
| Sharding | Splits dataset for parallel distributed processing |
| BPE tokenizer | Converts text → subword tokens → integer IDs |
| BOS / EOS tokens | Mark document start/end within packed sequences |
| Packing | Concatenate all tokens → reshape into fixed-length chunks |
| `max_seq_length` | Must match the model's context window at training time |
| Zero padding waste | Packing achieves ~100% GPU utilisation |
| Output | `./data/packaged_pretrain_dataset.parquet` → input to Notebook 5 |